1. Theory

Analytical solution to the regression problem + vector form



We are given feature matrix (Х), target vector (у)
Model: у = w1x1 + w2x2 + ... + b
w - weights
b - bias

Vector form: y = Xw 
y - column vector of targets
X - data matrix (each row is an observation, each column is a feature)
w - weight vector 

Why vector form? Scalar form describes one equation for one variable, e.g. y = 3x + 2
Vector form handles many equations simultaneously using matrix notation.

To minimize prediction error, we find the optimal w

We know about the loss function (Mean Squared Error), taking the derivative, setting to zero, and solving yields the Normal Equation:

![w](../pictures/веса.jpg)

What changes when L1 and L2 regularization are added to the loss function?

Regularization prevents overfitting by penalizing complex models. L2 reg(Ridge) penalizes the sum of squared weights.

What changes? Weights shrink toward zero (but rarely become exactly zero), the model becomes more stable and less sensitive to noise, all features are retained (no feature selection) 

![L2](../pictures/регуляризация.jpg)

What about L1 reg? It penalizes the sum of absolute weights and produces sparse solutions (many weights = 0). L2 has a quadratic penalty so reducing many small weights is "cheaper" than keeping one large weight. L1 has a linear penalty: if a feature contributes little, it's better to set its weight to exactly zero than to pay many small penalties

![L1](../pictures/L1.jpg)

How can linear models (Linear Regression, Ridge, Lasso) fit nonlinear relationships?

What is the problem with linear models? They can only build straight lines. But what if the data looks like a curve? For example, we look at the correlation between apartment area and price. At first, the price grows slowly, and then quickly - forming a curve. Then we can change not the model, but the data (we add squares and cubes of features, roots, logarithms, sines). And we feed those into the model. That is, we performed a nonlinear transformation of the features, but the model itself remains linear. We expanded the feature space

2) Data Preprocessing

Import libraries.
Read training and test parts

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
import ast
from collections import Counter
import re

In [2]:
train_df = pd.read_json('train.json')
test_df = pd.read_json('test.json')

3) Create additional features to improve model quality. Consider a column called "Features". It consists of a list of key characteristics of the current apartment.

Remove unused symbols from the column ([,], ', " and space).

Collect the results into one large list for the entire dataset.

In [3]:
print('train size:', train_df.shape)
print('test size:', test_df.shape)

# train columns
print(train_df.columns.tolist())

# features list
print(train_df['features'].head(3))

train size: (49352, 15)
test size: (74659, 14)
['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address', 'interest_level']
4    [Dining Room, Pre-War, Laundry in Building, Di...
6    [Doorman, Elevator, Laundry in Building, Dishw...
9    [Doorman, Elevator, Laundry in Building, Laund...
Name: features, dtype: object


In [4]:
def clean_features(features):
    if not isinstance(features, list):
        return []
    return [item.replace(' ', '') for item in features if isinstance(item, str)]
    
train_df['features_cleaned'] = train_df['features'].apply(clean_features)
test_df['features_cleaned'] = test_df['features'].apply(clean_features)

print("First 5 cleaned lists:")
for i in range(5):
    print(f"Row {i}: {train_df['features_cleaned'].iloc[i]}")

First 5 cleaned lists:
Row 0: ['DiningRoom', 'Pre-War', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'DogsAllowed', 'CatsAllowed']
Row 1: ['Doorman', 'Elevator', 'LaundryinBuilding', 'Dishwasher', 'HardwoodFloors', 'NoFee']
Row 2: ['Doorman', 'Elevator', 'LaundryinBuilding', 'LaundryinUnit', 'Dishwasher', 'HardwoodFloors']
Row 3: []
Row 4: ['Doorman', 'Elevator', 'FitnessCenter', 'LaundryinBuilding']


In [5]:
all_features = []

for index, row in train_df.iterrows():
    all_features.extend(row['features_cleaned'])

print(f"Total values: {len(all_features)}")

unique_features = set(all_features)
print(f"Unique values: {len(unique_features)}")

Total values: 267906
Unique values: 1548


Count the most popular features from our large list and select the top 20 for now.

In [6]:
feature_counter = Counter(all_features)
top_20 = feature_counter.most_common(20)

for i, (feature, number) in enumerate(top_20):
    if i == 0:
        print(f"('{feature}', {number})", end="")
    else:
        print(f",\n('{feature}', {number})", end="")

('Elevator', 25915),
('CatsAllowed', 23540),
('HardwoodFloors', 23527),
('DogsAllowed', 22035),
('Doorman', 20898),
('Dishwasher', 20426),
('NoFee', 18062),
('LaundryinBuilding', 16344),
('FitnessCenter', 13252),
('Pre-War', 9148),
('LaundryinUnit', 8738),
('RoofDeck', 6542),
('OutdoorSpace', 5268),
('DiningRoom', 5136),
('HighSpeedInternet', 4299),
('Balcony', 2992),
('SwimmingPool', 2730),
('LaundryInBuilding', 2593),
('NewConstruction', 2559),
('Terrace', 2283)

Now create 20 new features based on the top 20 values: 1 if the value is present in the "Features" column, otherwise 0.
Extend our feature set by adding "bathrooms" и "bedrooms",  and create a special variable "feature_list" with all feature names. Now we have 22 values. All models will be trained on these 22 features

In [7]:
top_20_features = [feature for feature, count in top_20]

for feature in top_20_features:
    column_name = f"has_{feature}"
    
    train_df[column_name] = train_df['features_cleaned'].apply(
        lambda lst: 1 if feature in lst else 0
    )
    
    test_df[column_name] = test_df['features_cleaned'].apply(
        lambda lst: 1 if feature in lst else 0
    )

print("Example of new columns:")
print([col for col in train_df.columns if col.startswith('has_')][:5])

Example of new columns:
['has_Elevator', 'has_CatsAllowed', 'has_HardwoodFloors', 'has_DogsAllowed', 'has_Doorman']


In [8]:
bathroom_col = 'bathrooms'
bedroom_col = 'bedrooms'

binary_features = [f'has_{feat}' for feat in top_20_features]
feature_list = binary_features + [bathroom_col, bedroom_col]

print(f"Total values: {len(feature_list)}")
print("feature_list:")
for i, feat in enumerate(feature_list, 1):
    print(f"{i:2}. {feat}")

Total values: 22
feature_list:
 1. has_Elevator
 2. has_CatsAllowed
 3. has_HardwoodFloors
 4. has_DogsAllowed
 5. has_Doorman
 6. has_Dishwasher
 7. has_NoFee
 8. has_LaundryinBuilding
 9. has_FitnessCenter
10. has_Pre-War
11. has_LaundryinUnit
12. has_RoofDeck
13. has_OutdoorSpace
14. has_DiningRoom
15. has_HighSpeedInternet
16. has_Balcony
17. has_SwimmingPool
18. has_LaundryInBuilding
19. has_NewConstruction
20. has_Terrace
21. bathrooms
22. bedrooms


In [9]:
x_train = train_df[feature_list]
y_train = train_df['price']

x_test = test_df[feature_list]

4. Model Implementation – Linear Regression

Implement a Python class for a linear regression algorithm with two basic methods - fit and predict. Use stochastic gradient descent (SGD) to find the optimal model weights

Theory

Gradient descent - an algorithm that step-by-step descends to the minimum of the error.

Non-stochastic (batch) gradient descent - take all training data (all apartments), compute the average error over all data, calculate the gradient (direction of improvement), and take a step in the opposite direction. Repeat. Pros: accurate direction and stability. Cons: slow, uses a lot of memory.

Stochastic (SGD) - take one point (one apartment), compute the error only on that one point, and calculate the gradient only from it. Take a step. Take the next point, and so on in a loop. Pros: fast, low memory usage. Cons: indirect path + lower accuracy.

Mini-batch gradient descent - the optimal and most popular option. Take a group of points (50 apartments), compute everything on them, and take a step.

In [10]:
class LinearRegressionSGD:

    def __init__(self, learning_rate=0.01, n_iter=1000, random_state=None):

        """
        Parameters:
        - learning_rate: step size
        - n_iter: number of iterations
        """

        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state
        self.weights = None  # coefficients for each feature
        self.bias = None  # bias
        self.loss_history = []  # loss history

    def fit(self, x, y):

        # initialize weights
        n_samples, n_features = x.shape    # n_samples=49352, n_features=222
        np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01 # initialize with small random numbers as weights
        self.bias = 0   # start with zero bias

        # SGD training
        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)  # shuffle data each time
            x_shuffled = x[indices]
            y_shuffled = y[indices]

            epoch_loss = 0

            for i in range(n_samples):
                # take one apartment
                x_i = x_shuffled[i]
                y_i = y_shuffled[i]

                y_pred = np.dot(x_i, self.weights) + self.bias
                # np.dot(x_i, self.weights) multiplies each apartment's feature by its weight and sums them

                error = y_pred - y_i

                dw = 2 * error * x_i  # gradient for ALL 22 weights
                db = 2 * error  # gradient for bias

                # update weights - move down along the gradient
                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                # MSE
                epoch_loss += error ** 2

            # average error per epoch
            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)

            # print progress every 100 epochs
            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{self.n_iter}, Loss: {avg_loss:.4f}")

    def predict(self, x):
        if self.weights is None:
            raise ValueError("Model not trained")
        
        return np.dot(x, self.weights) + self.bias

In [11]:
y_train_log = np.log1p(y_train)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train.values)

model_lin = LinearRegressionSGD(
    learning_rate=0.01,
    n_iter=500,
    random_state=21
)

model_lin.fit(x_train_scaled, y_train_log.values)

Epoch 100/500, Loss: 0.1087
Epoch 200/500, Loss: 0.1078
Epoch 300/500, Loss: 0.1084
Epoch 400/500, Loss: 0.1078
Epoch 500/500, Loss: 0.1072


In [ ]:
y_pred = model_lin.predict(x_train_scaled)

# metrics
mae = mean_absolute_error(y_train_log, y_pred)
rmse = np.sqrt(mean_squared_error(y_train_log, y_pred))
r2 = r2_score(y_train_log, y_pred)

print(f"MAE: {mae:.3f} k dollars")
print(f"RMSE: {rmse:.3f} k dollars")
print(f"R2: {r2:.4f}")

MEA: 0.235 k dollars
RMSE: 0.313 k dollars
R2: 0.4772


How did that happen? I applied log transformation to prices because there was a very large spread (from several tens to several million dollars) - in other words, I compressed the large values

R2 (R-squared) - coefficient of determination, ranges from 0 to 1. If 0 the model is essentially useless, predicts only the average price, if 1 predictions are perfect

What is a deterministic model? Make SGD deterministic.

Deterministic model - one that produces the same results every time given the same data. SGD is NOT deterministic because of np.random.premutation. We just need to modify the code slightly and add a shuffle parameter to the class

In [13]:
class LinearRegressionSGD:

    def __init__(self, learning_rate=0.01, n_iter=1000, random_state=None, shuffle=True):

        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.random_state = random_state
        self.shuffle = shuffle # True - stochastic, False - deterministic
        self.weights = None
        self.bias = None 
        self.loss_history = []

    def fit(self, x, y):

        n_samples, n_features = x.shape
        np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0

        for epoch in range(self.n_iter):
            if self.shuffle: 
                indices = np.random.permutation(n_samples)
                x_shuffled = x[indices]
                y_shuffled = y[indices]
            else: 
                x_shuffled = x
                y_shuffled = y

            epoch_loss = 0

            for i in range(n_samples):
                x_i = x_shuffled[i]
                y_i = y_shuffled[i]

                y_pred = np.dot(x_i, self.weights) + self.bias

                error = y_pred - y_i

                dw = 2 * error * x_i
                db = 2 * error

                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                epoch_loss += error ** 2

            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{self.n_iter}, Loss: {avg_loss:.4f}")

    def predict(self, x):
        if self.weights is None:
            raise ValueError("Model not trained")
        
        return np.dot(x, self.weights) + self.bias

Define the R² coefficient and implement a function to compute it.

In [14]:
def r2_score_custom(y_true, y_pred):

    # convert to arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    # sum of squared residuals (model errors)
    ss_res = np.sum((y_true - y_pred)**2)

    # total sum of squares (data variance)
    y_mean = np.mean(y_true)
    ss_tot = np.sum((y_true - y_mean)**2)

    # if data doesn't vary (ss_tot = 0)
    if ss_tot == 0:
        return 0.0
    
    return 1 - (ss_res / ss_tot)

y_true_test = y_train_log.values
y_pred_test = y_pred

r2_custom = r2_score_custom(y_true_test, y_pred_test)
r2_sklearn = r2_score_custom(y_true_test, y_pred_test)

print(f"My r2 function: {r2_custom:.6f}")
print(f"Sklearn r2: {r2_sklearn:.6f}")
print(f"Match? {abs(r2_custom - r2_sklearn) < 0.000001}")

My r2 function: 0.477218
Sklearn r2: 0.477218
Match? True


Initialize LinearRegression() from sklearn.linear_model, fit the model, and make predictions on train and test sets.
Compare quality metrics and ensure the difference is small (between my implementation and sklearn).
Save the metrics in a table with columns: model, train, test separately for MAE, RMSE table, and R2 table.

In [15]:
sk_model = LinearRegression()
sk_model.fit(x_train_scaled, y_train_log.values)

y_pred_sk_train = sk_model.predict(x_train_scaled)

x_test_scaled = scaler.transform(x_test.values)
y_test = test_df['price']
y_test_log = np.log1p(y_test)
y_pred_sk_test = sk_model.predict(x_test_scaled)


In [16]:
# our model on test

y_pred_our_test = model_lin.predict(x_test_scaled)

In [17]:
metrics_data = []
y_pred_our = model_lin.predict(x_train_scaled)

# our on train
metrics_data.append({
    'model': 'Our SGD',
    'dataset': 'train',
    'MAE': mean_absolute_error(y_train_log, y_pred_our),
    'RMSE': np.sqrt(mean_squared_error(y_train_log, y_pred_our)),
    'R2': r2_score(y_train_log, y_pred_our)
})

# our on test
metrics_data.append({
    'model': 'Our SGD',
    'dataset': 'test',
    'MAE': mean_absolute_error(y_test_log, y_pred_our_test),
    'RMSE': np.sqrt(mean_squared_error(y_test_log, y_pred_our_test)),
    'R2': r2_score(y_test_log, y_pred_our_test)
})

# sklearn - train
metrics_data.append({
    'model': 'Sklearn',
    'dataset': 'train',
    'MAE': mean_absolute_error(y_train_log, y_pred_sk_train),
    'RMSE': np.sqrt(mean_squared_error(y_train_log, y_pred_sk_train)),
    'R2': r2_score(y_train_log, y_pred_sk_train)
})


# sklearn - test
metrics_data.append({
    'model': 'Sklearn',
    'dataset': 'test',
    'MAE': mean_absolute_error(y_test_log, y_pred_sk_test),
    'RMSE': np.sqrt(mean_squared_error(y_test_log, y_pred_sk_test)),
    'R2': r2_score(y_test_log, y_pred_sk_test)
})

metrics_df = pd.DataFrame(metrics_data)
print("METRICS TABLE")
print(metrics_df.to_string(index=False))

METRICS TABLE
  model dataset      MAE     RMSE       R2
Our SGD   train 0.234642 0.312862 0.477218
Our SGD    test 0.236528 0.337335 0.390483
Sklearn   train 0.202809 0.277188 0.589640
Sklearn    test 0.204434 0.308484 0.490285


SGD works fine (train/test difference is normal) but slightly weaker than sklearn. We can increase number of epochs OR adapt the learning rate.

Regularized models implementation - Ridge, Lasso, ElasticNet

Implement алгоритмы Ridge, Lasso и ElasticNet: extend the loss function with L2, L1, and both regularizations.

In [18]:
# Linear regression with L2 regularization (Ridge)
class LinearRegressionRidge:

    def __init__(self, learning_rate=0.01, n_iter=1000, lambda_reg=1.0, random_state=None):

        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.lambda_reg = lambda_reg  # regularization coefficient
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit (self, x, y):
        n_samples, n_features = x.shape
        np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01  # 22 small random numbers
        self.bias = 0

        for epoch in range (self.n_iter):
            indices = np.random.permutation(n_samples)  # shuffle
            x_shuffled = x[indices]
            y_shuffled = y[indices]

            epoch_loss = 0

            for i in range (n_samples):
                x_i = x_shuffled[i]   # features of one apartment
                y_i = y_shuffled[i]  # its price

                y_pred = np.dot(x_i, self.weights) + self.bias  # multiply each feature by its weight and sum
                error = y_pred - y_i

                dw = 2 * error * x_i + 2 * self.lambda_reg * self.weights
                db = 2 * error

                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                mse_loss = error ** 2
                l2_penalty = self.lambda_reg * np.sum(self.weights ** 2)  # penalty for all weights
                epoch_loss += mse_loss + l2_penalty

            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{self.n_iter}, Loss: {avg_loss:.4f}")

    
    def predict (self, x):
        if self.weights is None:
            raise ValueError ("Model not trained")
        return np.dot(x, self.weights) + self.bias


In [19]:
# L1 regularization (Lasso)

class LinearRegressionLasso:

    def __init__(self, learning_rate=0.01, n_iter=1000, lambda_reg=1.0, random_state=None):
        
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.lambda_reg = lambda_reg
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.loss_history = []


    def fit (self, x, y):
        n_samples, n_features = x.shape
        np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0

        for epoch in range (self.n_iter):
            indices = np.random.permutation(n_samples)
            x_shuffled = x[indices]
            y_shuffled = y[indices]

            epoch_loss = 0

            for i in range (n_samples):
                x_i = x_shuffled[i]
                y_i = y_shuffled[i]

                y_pred = np.dot(x_i, self.weights) + self.bias
                error = y_pred - y_i

                dw = 2 * error * x_i + self.lambda_reg * np.sign(self.weights)  # penalizes not the square of the weight but the absolute value - weights become zero
                db = 2 * error

                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                mse_loss = error ** 2
                l1_penalty = self.lambda_reg * np.sum(np.abs(self.weights))
                epoch_loss += mse_loss + l1_penalty

            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{self.n_iter}, Loss: {avg_loss:.4f}")

    def predict(self, x):
        if self.weights is None:
            raise ValueError("Model not trained")
        return np.dot(x, self.weights) + self.bias

In [20]:
# ElasticNet (L1 + L2)

class LinearRegressionElasticNet:
    def __init__(self, learning_rate=0.01, n_iter=1000, lambda_1=1.0, lambda_2=1.0, random_state=None):
        
        self.learning_rate = learning_rate
        self.n_iter = n_iter
        self.lambda_1 = lambda_1  # coefficient L1
        self.lambda_2 = lambda_2  # coefficient L2
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.loss_history = []

    def fit(self, x, y):
        n_samples, n_features = x.shape
        np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0

        for epoch in range(self.n_iter):
            indices = np.random.permutation(n_samples)
            x_shuffled = x[indices]
            y_shuffled = y[indices]

            epoch_loss = 0

            for i in range(n_samples):
                x_i = x_shuffled[i]
                y_i = y_shuffled[i]

                y_pred = np.dot(x_i, self.weights) + self.bias
                error = y_pred - y_i

                dw = (2 * error * x_i 
                      + self.lambda_1 * np.sign(self.weights)   # L1 penalty
                      + 2 * self.lambda_2 * self.weights)      # L2 penalty
                db = 2 * error

                self.weights -= self.learning_rate * dw
                self.bias -= self.learning_rate * db

                mse_loss = error ** 2
                l1_penalty = self.lambda_1 * np.sum(np.abs(self.weights))
                l2_penalty = self.lambda_2 * np.sum(self.weights ** 2)
                epoch_loss += mse_loss + l1_penalty + l2_penalty

            avg_loss = epoch_loss / n_samples
            self.loss_history.append(avg_loss)

            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{self.n_iter}, Loss: {avg_loss:.4f}")

    def predict(self, x):
        if self.weights is None:
            raise ValueError("Model not trained")
        return np.dot(x, self.weights) + self.bias

Make predictions with my algorithm and evaluate the model using MAE, RMSE и R2 metrics.

In [21]:
ridge_model = LinearRegressionRidge(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=1.0,
    random_state=21
)

ridge_model.fit(x_train_scaled, y_train_log.values)

y_pred_ridge_train = ridge_model.predict(x_train_scaled)
y_pred_ridge_test = ridge_model.predict(x_test_scaled)

Epoch 100/200, Loss: 0.1477
Epoch 200/200, Loss: 0.1477


In [22]:
# train
mae_ridge_train = mean_absolute_error(y_train_log, y_pred_ridge_train)
rmse_ridge_train = np.sqrt(mean_squared_error(y_train_log, y_pred_ridge_train))
r2_ridge_train = r2_score(y_train_log, y_pred_ridge_train)

# test
mae_ridge_test = mean_absolute_error(y_test_log, y_pred_ridge_test)
rmse_ridge_test = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_test))
r2_ridge_test = r2_score(y_test_log, y_pred_ridge_test)

print(f"{'Model':<15} {'Dataset':<8} {'MAE':<10} {'RMSE':<10} {'R2':<10}")

print(f"{'Ridge (Ours)':<15} {'train':<8} {mae_ridge_train:<10.4f} {rmse_ridge_train:<10.4f} {r2_ridge_train:<10.4f}")
print(f"{'Ridge (Ours)':<15} {'test':<8} {mae_ridge_test:<10.4f} {rmse_ridge_test:<10.4f} {r2_ridge_test:<10.4f}")

Model           Dataset  MAE        RMSE       R2        
Ridge (Ours)    train    0.2607     0.3445     0.3661    
Ridge (Ours)    test     0.2623     0.3521     0.3359    


In [23]:
lasso_model = LinearRegressionLasso(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=0.1,
    random_state=21
)

lasso_model.fit(x_train_scaled, y_train_log.values)

y_pred_lasso_train = lasso_model.predict(x_train_scaled)
y_pred_lasso_test = lasso_model.predict(x_test_scaled)

Epoch 100/200, Loss: 0.1578
Epoch 200/200, Loss: 0.1581


In [24]:
mae_lasso_train = mean_absolute_error(y_train_log, y_pred_lasso_train)
rmse_lasso_train = np.sqrt(mean_squared_error(y_train_log, y_pred_lasso_train))
r2_lasso_train = r2_score(y_train_log, y_pred_lasso_train)

mae_lasso_test = mean_absolute_error(y_test_log, y_pred_lasso_test)
rmse_lasso_test = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_test))
r2_lasso_test = r2_score(y_test_log, y_pred_lasso_test)

print(f"{'Model':<15} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")

print(f"{'Lasso (Ours)':<15} {'train':<8} {mae_lasso_train:<12.4f} {rmse_lasso_train:<12.4f} {r2_lasso_train:<12.4f}")
print(f"{'Lasso (Ours)':<15} {'test':<8} {mae_lasso_test:<12.4f} {rmse_lasso_test:<12.4f} {r2_lasso_test:<12.4f}")

Model           Dataset  MAE          RMSE         R2          
Lasso (Ours)    train    0.2455       0.3241       0.4392      
Lasso (Ours)    test     0.2474       0.3494       0.3462      


In [25]:
elastic_model = LinearRegressionElasticNet(
    learning_rate=0.01,
    n_iter=200,
    lambda_1=0.05,
    lambda_2=0.05,
    random_state=21
)

elastic_model.fit(x_train_scaled, y_train_log.values)

y_pred_elastic_train = elastic_model.predict(x_train_scaled)
y_pred_elastic_test = elastic_model.predict(x_test_scaled)

mae_elastic_train = mean_absolute_error(y_train_log, y_pred_elastic_train)
rmse_elastic_train = np.sqrt(mean_squared_error(y_train_log, y_pred_elastic_train))
r2_elastic_train = r2_score(y_train_log, y_pred_elastic_train)

mae_elastic_test = mean_absolute_error(y_test_log, y_pred_elastic_test)
rmse_elastic_test = np.sqrt(mean_squared_error(y_test_log, y_pred_elastic_test))
r2_elastic_test = r2_score(y_test_log, y_pred_elastic_test)

print(f"{'Model':<15} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'ElasticNet':<15} {'train':<8} {mae_elastic_train:<12.4f} {rmse_elastic_train:<12.4f} {r2_elastic_train:<12.4f}")
print(f"{'ElasticNet':<15} {'test':<8} {mae_elastic_test:<12.4f} {rmse_elastic_test:<12.4f} {r2_elastic_test:<12.4f}")

Epoch 100/200, Loss: 0.1377
Epoch 200/200, Loss: 0.1378
Model           Dataset  MAE          RMSE         R2          
ElasticNet      train    0.2426       0.3201       0.4527      
ElasticNet      test     0.2444       0.3457       0.3599      


Initialize Ridge(), Lasso() и ElasticNet() from sklearn.linear_model, fit the model, and make predictions on train and test sets.

In [26]:
ridge_sk = Ridge(alpha=0.1)  # alpha = lambda_reg
ridge_sk.fit(x_train_scaled, y_train_log.values)

y_pred_ridge_sk_train = ridge_sk.predict(x_train_scaled)
y_pred_ridge_sk_test = ridge_sk.predict(x_test_scaled)

mae_ridge_sk_train = mean_absolute_error(y_train_log, y_pred_ridge_sk_train)
rmse_ridge_sk_train = np.sqrt(mean_squared_error(y_train_log, y_pred_ridge_sk_train))
r2_ridge_sk_train = r2_score(y_train_log, y_pred_ridge_sk_train)

mae_ridge_sk_test = mean_absolute_error(y_test_log, y_pred_ridge_sk_test)
rmse_ridge_sk_test = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_sk_test))
r2_ridge_sk_test = r2_score(y_test_log, y_pred_ridge_sk_test)

print(f"{'Model':<20} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'Our Ridge':<20} {'train':<8} {mae_ridge_train:<12.4f} {rmse_ridge_train:<12.4f} {r2_ridge_train:<12.4f}")
print(f"{'Our Ridge':<20} {'test':<8} {mae_ridge_test:<12.4f} {rmse_ridge_test:<12.4f} {r2_ridge_test:<12.4f}")
print(f"{'Sklearn Ridge':<20} {'train':<8} {mae_ridge_sk_train:<12.4f} {rmse_ridge_sk_train:<12.4f} {r2_ridge_sk_train:<12.4f}")
print(f"{'Sklearn Ridge':<20} {'test':<8} {mae_ridge_sk_test:<12.4f} {rmse_ridge_sk_test:<12.4f} {r2_ridge_sk_test:<12.4f}")


Model                Dataset  MAE          RMSE         R2          
Our Ridge            train    0.2607       0.3445       0.3661      
Our Ridge            test     0.2623       0.3521       0.3359      
Sklearn Ridge        train    0.2028       0.2772       0.5896      
Sklearn Ridge        test     0.2044       0.3085       0.4903      


In [27]:
lasso_sk = Lasso(alpha=0.1)
lasso_sk.fit(x_train_scaled, y_train_log.values)

y_pred_lasso_sk_train = lasso_sk.predict(x_train_scaled)
y_pred_lasso_sk_test = lasso_sk.predict(x_test_scaled)

mae_lasso_sk_train = mean_absolute_error(y_train_log, y_pred_lasso_sk_train)
rmse_lasso_sk_train = np.sqrt(mean_squared_error(y_train_log, y_pred_lasso_sk_train))
r2_lasso_sk_train = r2_score(y_train_log, y_pred_lasso_sk_train)

mae_lasso_sk_test = mean_absolute_error(y_test_log, y_pred_lasso_sk_test)
rmse_lasso_sk_test = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_sk_test))
r2_lasso_sk_test = r2_score(y_test_log, y_pred_lasso_sk_test)

print(f"{'Model':<20} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'Our Lasso':<20} {'train':<8} {mae_lasso_train:<12.4f} {rmse_lasso_train:<12.4f} {r2_lasso_train:<12.4f}")
print(f"{'Our Lasso':<20} {'test':<8} {mae_lasso_test:<12.4f} {rmse_lasso_test:<12.4f} {r2_lasso_test:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'train':<8} {mae_lasso_sk_train:<12.4f} {rmse_lasso_sk_train:<12.4f} {r2_lasso_sk_train:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'test':<8} {mae_lasso_sk_test:<12.4f} {rmse_lasso_sk_test:<12.4f} {r2_lasso_sk_test:<12.4f}")

Model                Dataset  MAE          RMSE         R2          
Our Lasso            train    0.2455       0.3241       0.4392      
Our Lasso            test     0.2474       0.3494       0.3462      
Sklearn Lasso        train    0.2392       0.3214       0.4483      
Sklearn Lasso        test     0.2407       0.3401       0.3806      


In [28]:
elastic_sk = ElasticNet(alpha=0.05, l1_ratio=0.5)
elastic_sk.fit(x_train_scaled, y_train_log.values)

y_pred_elastic_sk_train = elastic_sk.predict(x_train_scaled)
y_pred_elastic_sk_test = elastic_sk.predict(x_test_scaled)

mae_elastic_sk_train = mean_absolute_error(y_train_log, y_pred_elastic_sk_train)
rmse_elastic_sk_train = np.sqrt(mean_squared_error(y_train_log, y_pred_elastic_sk_train))
r2_elastic_sk_train = r2_score(y_train_log, y_pred_elastic_sk_train)

mae_elastic_sk_test = mean_absolute_error(y_test_log, y_pred_elastic_sk_test)
rmse_elastic_sk_test = np.sqrt(mean_squared_error(y_test_log, y_pred_elastic_sk_test))
r2_elastic_sk_test = r2_score(y_test_log, y_pred_elastic_sk_test)

print(f"{'Model':<20} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'Our Elastic':<20} {'train':<8} {mae_elastic_train:<12.4f} {rmse_elastic_train:<12.4f} {r2_elastic_train:<12.4f}")
print(f"{'Our Elastic':<20} {'test':<8} {mae_elastic_test:<12.4f} {rmse_elastic_test:<12.4f} {r2_elastic_test:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'train':<8} {mae_elastic_sk_train:<12.4f} {rmse_elastic_sk_train:<12.4f} {r2_elastic_sk_train:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'test':<8} {mae_elastic_sk_test:<12.4f} {rmse_elastic_sk_test:<12.4f} {r2_elastic_sk_test:<12.4f}")

Model                Dataset  MAE          RMSE         R2          
Our Elastic          train    0.2426       0.3201       0.4527      
Our Elastic          test     0.2444       0.3457       0.3599      
Sklearn Elastic      train    0.2100       0.2858       0.5638      
Sklearn Elastic      test     0.2115       0.3145       0.4701      


In [56]:
print("MAE METRICS")

print(f"{'Model':<20} {'Dataset':<8} {'MAE':<12}")

print(f"{'Our Ridge':<20} {'train':<8} {mae_ridge_train:<12.4f}")
print(f"{'Our Ridge':<20} {'test':<8} {mae_ridge_test:<12.4f}")
print(f"{'Our Lasso':<20} {'train':<8} {mae_lasso_train:<12.4f}")
print(f"{'Our Lasso':<20} {'test':<8} {mae_lasso_test:<12.4f}")
print(f"{'Our Elastic':<20} {'train':<8} {mae_elastic_train:<12.4f}")
print(f"{'Our Elastic':<20} {'test':<8} {mae_elastic_test:<12.4f}")

print(f"{'Sklearn Ridge':<20} {'train':<8} {mae_ridge_sk_train:<12.4f}")
print(f"{'Sklearn Ridge':<20} {'test':<8} {mae_ridge_sk_test:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'train':<8} {mae_lasso_sk_train:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'test':<8} {mae_lasso_sk_test:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'train':<8} {mae_elastic_sk_train:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'test':<8} {mae_elastic_sk_test:<12.4f}")

MAE METRICS
Model                Dataset  MAE         
Our Ridge            train    0.2607      
Our Ridge            test     0.2623      
Our Lasso            train    0.2455      
Our Lasso            test     0.2474      
Our Elastic          train    0.2426      
Our Elastic          test     0.2444      
Sklearn Ridge        train    0.2028      
Sklearn Ridge        test     0.2044      
Sklearn Lasso        train    0.2392      
Sklearn Lasso        test     0.2407      
Sklearn Elastic      train    0.2100      
Sklearn Elastic      test     0.2115      


In [30]:
print("RMSE METRICS")

print(f"{'Model':<20} {'Dataset':<8} {'RMSE':<12}")

print(f"{'Our Ridge':<20} {'train':<8} {rmse_ridge_train:<12.4f}")
print(f"{'Our Ridge':<20} {'test':<8} {rmse_ridge_test:<12.4f}")
print(f"{'Our Lasso':<20} {'train':<8} {rmse_lasso_train:<12.4f}")
print(f"{'Our Lasso':<20} {'test':<8} {rmse_lasso_test:<12.4f}")
print(f"{'Our Elastic':<20} {'train':<8} {rmse_elastic_train:<12.4f}")
print(f"{'Our Elastic':<20} {'test':<8} {rmse_elastic_test:<12.4f}")

print(f"{'Sklearn Ridge':<20} {'train':<8} {rmse_ridge_sk_train:<12.4f}")
print(f"{'Sklearn Ridge':<20} {'test':<8} {rmse_ridge_sk_test:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'train':<8} {rmse_lasso_sk_train:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'test':<8} {rmse_lasso_sk_test:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'train':<8} {rmse_elastic_sk_train:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'test':<8} {rmse_elastic_sk_test:<12.4f}")


RMSE METRICS
Model                Dataset  RMSE        
Our Ridge            train    0.3445      
Our Ridge            test     0.3521      
Our Lasso            train    0.3241      
Our Lasso            test     0.3494      
Our Elastic          train    0.3201      
Our Elastic          test     0.3457      
Sklearn Ridge        train    0.2772      
Sklearn Ridge        test     0.3085      
Sklearn Lasso        train    0.3214      
Sklearn Lasso        test     0.3401      
Sklearn Elastic      train    0.2858      
Sklearn Elastic      test     0.3145      


In [31]:
print("R² METRICS")

print(f"{'Model':<20} {'Dataset':<8} {'R2':<12}")

print(f"{'Our Ridge':<20} {'train':<8} {r2_ridge_train:<12.4f}")
print(f"{'Our Ridge':<20} {'test':<8} {r2_ridge_test:<12.4f}")
print(f"{'Our Lasso':<20} {'train':<8} {r2_lasso_train:<12.4f}")
print(f"{'Our Lasso':<20} {'test':<8} {r2_lasso_test:<12.4f}")
print(f"{'Our Elastic':<20} {'train':<8} {r2_elastic_train:<12.4f}")
print(f"{'Our Elastic':<20} {'test':<8} {r2_elastic_test:<12.4f}")

print(f"{'Sklearn Ridge':<20} {'train':<8} {r2_ridge_sk_train:<12.4f}")
print(f"{'Sklearn Ridge':<20} {'test':<8} {r2_ridge_sk_test:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'train':<8} {r2_lasso_sk_train:<12.4f}")
print(f"{'Sklearn Lasso':<20} {'test':<8} {r2_lasso_sk_test:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'train':<8} {r2_elastic_sk_train:<12.4f}")
print(f"{'Sklearn Elastic':<20} {'test':<8} {r2_elastic_sk_test:<12.4f}")

R² METRICS
Model                Dataset  R2          
Our Ridge            train    0.3661      
Our Ridge            test     0.3359      
Our Lasso            train    0.4392      
Our Lasso            test     0.3462      
Our Elastic          train    0.4527      
Our Elastic          test     0.3599      
Sklearn Ridge        train    0.5896      
Sklearn Ridge        test     0.4903      
Sklearn Lasso        train    0.4483      
Sklearn Lasso        test     0.3806      
Sklearn Elastic      train    0.5638      
Sklearn Elastic      test     0.4701      


6. Normalization of features

why is it even needed? for example, we have many features with different scales. there might be 1 or 2 bathrooms, bedrooms could be 1-5, and price is generally from a hundred thousand to several million.
in such a situation, the feature with a large range (price) will dominate, and small features like the number of bathrooms or elevators will practically not affect training.

Normalization is mandatory when working with gradient descent and features are on different scales. Also, when using regularization (L1/L2), because it will work unevenly. And when comparing model weights, since you cannot compare weights from different scales.

you can NOT normalize when working with Random Forest (XGBoost) - they are not sensitive to scale. or when all features are already on the same scale. or in situations where it is fundamentally important to understand the feature in its original units. 

Consider the first of the classical normalization methods - MinMaxScaler. Mathematical formula for this method:

![Min_Max](../pictures/norm.jpg)

Implement our own function or class for MinMaxScaler feature normalization.

In [32]:
class MyMinMaxScaler:

    def __init__(self):
        self.min_ = None
        self.max_ = None
        self.range_ = None   # max - min

    def fit(self, x):
        x = np.array(x)
        self.min_ = x.min(axis=0)
        self.max_= x.max(axis=0)
        self.range_ = self.max_ - self.min_

        self.range_[self.range_ == 0] = 1

        return self
    
    def transform(self, x):
        x = np.array(x)
        return (x - self.min_) / self.range_


Initialize MinMaxScaler() from sklearn.preprocessing.

In [33]:
sk_mm_scaler = MinMaxScaler()

Compare feature normalization using our own method and sklearn.

In [34]:
# our

our_scaler = MyMinMaxScaler()
out_scaler = our_scaler.fit(x_train.values)
x_our_norm = our_scaler.transform(x_train.values)

# sklearn

x_sk_norm = sk_mm_scaler.fit_transform(x_train.values)

are_equal = np.allclose(x_our_norm, x_sk_norm)

print(f"Our normalization: {x_our_norm.shape}")
print(f"Sklearn normalization: {x_sk_norm.shape}")
print(f"Match? {are_equal}")

if not are_equal:
    max_diff = np.max(np.abs(x_our_norm - x_sk_norm))
    print(f"Max difference: {max_diff:.10f}")

Our normalization: (49352, 22)
Sklearn normalization: (49352, 22)
Match? True


Repeat the steps for another normalization method - StandardScaler.

![standart_scaler](../pictures/stanscaler.jpg)

In [35]:
class MyStandartScaler:

    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        x = np.array(x)
        self.mean_ = x.mean(axis=0)
        self.std_ = x.std(axis=0)

        self.std_[self.std_ == 0] = 1
        return self
    
    def transform(self, x):
        x = np.array(x)
        return (x - self.mean_) / self.std_

In [36]:
# our
our_stanscaler = MyStandartScaler()
our_stanscaler.fit(x_train.values)
x_our_stannorm = our_stanscaler.transform(x_train.values)

#sklearn
sk_stanscaler = StandardScaler()
x_sk_stannorm = sk_stanscaler.fit_transform(x_train.values)

are_equal = np.allclose(x_our_stannorm, x_sk_stannorm)

print(f"Our normalization: {x_our_stannorm.shape}")
print(f"Sklearn normalization: {x_sk_stannorm.shape}")
print(f"Match? {are_equal}")

if not are_equal:
    max_diff = np.max(np.abs(x_our_stannorm - x_sk_stannorm))
    print(f"Max difference: {max_diff:.10f}")

Our normalization: (49352, 22)
Sklearn normalization: (49352, 22)
Match? True


7. Fitting Custom and Sklearn Models to Normalized Data

Fit all models - Linear Regression, Ridge, Lasso, and ElasticNet using MinMaxScaler.

In [37]:
mm_scaler = MinMaxScaler()
x_train_mm = mm_scaler.fit_transform(x_train.values)
x_test_mm = mm_scaler.transform(x_test.values)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"X_train_mm shape: {x_train_mm.shape}")
print(f"X_test_mm shape: {x_test_mm.shape}")
print(f"y_train_log shape: {y_train_log.shape}")

X_train_mm shape: (49352, 22)
X_test_mm shape: (74659, 22)
y_train_log shape: (49352,)


In [38]:
# SGD

sgd_mm = LinearRegressionSGD(
    learning_rate=0.01,
    n_iter=200,
    random_state=21
)

sgd_mm.fit(x_train_mm, y_train_log.values)

y_pred_mm_train = sgd_mm.predict(x_train_mm)
y_pred_mm_test = sgd_mm.predict(x_test_mm)

mae_mm_train = mean_absolute_error(y_train_log, y_pred_mm_train)
rmse_mm_train = np.sqrt(mean_squared_error(y_train_log, y_pred_mm_train))
r2_mm_train = r2_score(y_train_log, y_pred_mm_train)

mae_mm_test = mean_absolute_error(y_test_log, y_pred_mm_test)
rmse_mm_test = np.sqrt(mean_squared_error(y_test_log, y_pred_mm_test))
r2_mm_test = r2_score(y_test_log, y_pred_mm_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_mm_train:<12.4f} {rmse_mm_train:<12.4f} {r2_mm_train:<12.4f}")
print(f"{'test':<10} {mae_mm_test:<12.4f} {rmse_mm_test:<12.4f} {r2_mm_test:<12.4f}")


Epoch 100/200, Loss: 0.0813
Epoch 200/200, Loss: 0.0812
Dataset    MAE          RMSE         R2          
train      0.2105       0.2832       0.5716      
test       0.2119       0.3143       0.4708      


In [39]:
# Ridge

ridge_mm = LinearRegressionRidge(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=0.1,
    random_state=21
)

ridge_mm.fit(x_train_mm, y_train_log.values)

y_pred_ridge_mm_train = ridge_mm.predict(x_train_mm)
y_pred_ridge_mm_test = ridge_mm.predict(x_test_mm)

mae_ridge_mm_train = mean_absolute_error(y_train_log, y_pred_ridge_mm_train)
rmse_ridge_mm_train = np.sqrt(mean_squared_error(y_train_log, y_pred_ridge_mm_train))
r2_ridge_mm_train = r2_score(y_train_log, y_pred_ridge_mm_train)

mae_ridge_mm_test = mean_absolute_error(y_test_log, y_pred_ridge_mm_test)
rmse_ridge_mm_test = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_mm_test))
r2_ridge_mm_test = r2_score(y_test_log, y_pred_ridge_mm_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_ridge_mm_train:<12.4f} {rmse_ridge_mm_train:<12.4f} {r2_ridge_mm_train:<12.4f}")
print(f"{'test':<10} {mae_ridge_mm_test:<12.4f} {rmse_ridge_mm_test:<12.4f} {r2_ridge_mm_test:<12.4f}")

Epoch 100/200, Loss: 0.1572
Epoch 200/200, Loss: 0.1572
Dataset    MAE          RMSE         R2          
train      0.2734       0.3742       0.2522      
test       0.2740       0.3734       0.2531      


In [40]:
lasso_mm = LinearRegressionLasso(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=0.01,
    random_state=21
)

lasso_mm.fit(x_train_mm, y_train_log.values)

y_pred_lasso_mm_train = lasso_mm.predict(x_train_mm)
y_pred_lasso_mm_test = lasso_mm.predict(x_test_mm)

mae_lasso_mm_train = mean_absolute_error(y_train_log, y_pred_lasso_mm_train)
rmse_lasso_mm_train = np.sqrt(mean_squared_error(y_train_log, y_pred_lasso_mm_train))
r2_lasso_mm_train = r2_score(y_train_log, y_pred_lasso_mm_train)

mae_lasso_mm_test = mean_absolute_error(y_test_log, y_pred_lasso_mm_test)
rmse_lasso_mm_test = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_mm_test))
r2_lasso_mm_test = r2_score(y_test_log, y_pred_lasso_mm_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_lasso_mm_train:<12.4f} {rmse_lasso_mm_train:<12.4f} {r2_lasso_mm_train:<12.4f}")
print(f"{'test':<10} {mae_lasso_mm_test:<12.4f} {rmse_lasso_mm_test:<12.4f} {r2_lasso_mm_test:<12.4f}")

Epoch 100/200, Loss: 0.1227
Epoch 200/200, Loss: 0.1228
Dataset    MAE          RMSE         R2          
train      0.2239       0.3023       0.5121      
test       0.2252       0.3054       0.5004      


In [41]:
# elasticnet

elastic_mm = LinearRegressionElasticNet(
    learning_rate=0.01,
    n_iter=200,
    lambda_1=0.01,    
    lambda_2=0.01,    
    random_state=21
)

elastic_mm.fit(x_train_mm, y_train_log.values)

y_pred_elastic_mm_train = elastic_mm.predict(x_train_mm)
y_pred_elastic_mm_test = elastic_mm.predict(x_test_mm)

mae_elastic_mm_train = mean_absolute_error(y_train_log, y_pred_elastic_mm_train)
rmse_elastic_mm_train = np.sqrt(mean_squared_error(y_train_log, y_pred_elastic_mm_train))
r2_elastic_mm_train = r2_score(y_train_log, y_pred_elastic_mm_train)

mae_elastic_mm_test = mean_absolute_error(y_test_log, y_pred_elastic_mm_test)
rmse_elastic_mm_test = np.sqrt(mean_squared_error(y_test_log, y_pred_elastic_mm_test))
r2_elastic_mm_test = r2_score(y_test_log, y_pred_elastic_mm_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_elastic_mm_train:<12.4f} {rmse_elastic_mm_train:<12.4f} {r2_elastic_mm_train:<12.4f}")
print(f"{'test':<10} {mae_elastic_mm_test:<12.4f} {rmse_elastic_mm_test:<12.4f} {r2_elastic_mm_test:<12.4f}")

Epoch 100/200, Loss: 0.1397
Epoch 200/200, Loss: 0.1397
Dataset    MAE          RMSE         R2          
train      0.2395       0.3263       0.4314      
test       0.2406       0.3266       0.4288      


Fit all models -  линейную регрессию, Ridge, Lasso и ElasticNet using StandardScaler.

In [42]:
std_scaler = StandardScaler()
x_train_std = std_scaler.fit_transform(x_train.values)
x_test_std = std_scaler.transform(x_test.values)

print(f"X_train_std shape: {x_train_std.shape}")
print(f"X_test_std shape: {x_test_std.shape}")

X_train_std shape: (49352, 22)
X_test_std shape: (74659, 22)


In [43]:
# SGD 
sgd_std = LinearRegressionSGD(
    learning_rate=0.01,
    n_iter=200,
    random_state=21
)

sgd_std.fit(x_train_std, y_train_log.values)

y_pred_sgd_std_train = sgd_std.predict(x_train_std)
y_pred_sgd_std_test = sgd_std.predict(x_test_std)

mae_sgd_std_train = mean_absolute_error(y_train_log, y_pred_sgd_std_train)
rmse_sgd_std_train = np.sqrt(mean_squared_error(y_train_log, y_pred_sgd_std_train))
r2_sgd_std_train = r2_score(y_train_log, y_pred_sgd_std_train)

mae_sgd_std_test = mean_absolute_error(y_test_log, y_pred_sgd_std_test)
rmse_sgd_std_test = np.sqrt(mean_squared_error(y_test_log, y_pred_sgd_std_test))
r2_sgd_std_test = r2_score(y_test_log, y_pred_sgd_std_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_sgd_std_train:<12.4f} {rmse_sgd_std_train:<12.4f} {r2_sgd_std_train:<12.4f}")
print(f"{'test':<10} {mae_sgd_std_test:<12.4f} {rmse_sgd_std_test:<12.4f} {r2_sgd_std_test:<12.4f}")

Epoch 100/200, Loss: 0.1087
Epoch 200/200, Loss: 0.1078
Dataset    MAE          RMSE         R2          
train      0.2492       0.3269       0.4294      
test       0.2507       0.3583       0.3124      


In [44]:
# Ridge
ridge_std = LinearRegressionRidge(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=0.1,
    random_state=21
)

ridge_std.fit(x_train_std, y_train_log.values)

y_pred_ridge_std_train = ridge_std.predict(x_train_std)
y_pred_ridge_std_test = ridge_std.predict(x_test_std)

mae_ridge_std_train = mean_absolute_error(y_train_log, y_pred_ridge_std_train)
rmse_ridge_std_train = np.sqrt(mean_squared_error(y_train_log, y_pred_ridge_std_train))
r2_ridge_std_train = r2_score(y_train_log, y_pred_ridge_std_train)

mae_ridge_std_test = mean_absolute_error(y_test_log, y_pred_ridge_std_test)
rmse_ridge_std_test = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_std_test))
r2_ridge_std_test = r2_score(y_test_log, y_pred_ridge_std_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_ridge_std_train:<12.4f} {rmse_ridge_std_train:<12.4f} {r2_ridge_std_train:<12.4f}")
print(f"{'test':<10} {mae_ridge_std_test:<12.4f} {rmse_ridge_std_test:<12.4f} {r2_ridge_std_test:<12.4f}")

Epoch 100/200, Loss: 0.1134
Epoch 200/200, Loss: 0.1129
Dataset    MAE          RMSE         R2          
train      0.2500       0.3280       0.4254      
test       0.2517       0.3539       0.3291      


In [45]:
# Lasso
lasso_std = LinearRegressionLasso(
    learning_rate=0.01,
    n_iter=200,
    lambda_reg=0.01,
    random_state=21
)

lasso_std.fit(x_train_std, y_train_log.values)

y_pred_lasso_std_train = lasso_std.predict(x_train_std)
y_pred_lasso_std_test = lasso_std.predict(x_test_std)

mae_lasso_std_train = mean_absolute_error(y_train_log, y_pred_lasso_std_train)
rmse_lasso_std_train = np.sqrt(mean_squared_error(y_train_log, y_pred_lasso_std_train))
r2_lasso_std_train = r2_score(y_train_log, y_pred_lasso_std_train)

mae_lasso_std_test = mean_absolute_error(y_test_log, y_pred_lasso_std_test)
rmse_lasso_std_test = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_std_test))
r2_lasso_std_test = r2_score(y_test_log, y_pred_lasso_std_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_lasso_std_train:<12.4f} {rmse_lasso_std_train:<12.4f} {r2_lasso_std_train:<12.4f}")
print(f"{'test':<10} {mae_lasso_std_test:<12.4f} {rmse_lasso_std_test:<12.4f} {r2_lasso_std_test:<12.4f}")

Epoch 100/200, Loss: 0.1147
Epoch 200/200, Loss: 0.1140
Dataset    MAE          RMSE         R2          
train      0.2462       0.3234       0.4413      
test       0.2478       0.3545       0.3268      


In [46]:
elastic_std = LinearRegressionElasticNet(
    learning_rate=0.01,
    n_iter=200,
    lambda_1=0.01,
    lambda_2=0.01,
    random_state=21
)

elastic_std.fit(x_train_std, y_train_log.values)

y_pred_elastic_std_train = elastic_std.predict(x_train_std)
y_pred_elastic_std_test = elastic_std.predict(x_test_std)

mae_elastic_std_train = mean_absolute_error(y_train_log, y_pred_elastic_std_train)
rmse_elastic_std_train = np.sqrt(mean_squared_error(y_train_log, y_pred_elastic_std_train))
r2_elastic_std_train = r2_score(y_train_log, y_pred_elastic_std_train)

mae_elastic_std_test = mean_absolute_error(y_test_log, y_pred_elastic_std_test)
rmse_elastic_std_test = np.sqrt(mean_squared_error(y_test_log, y_pred_elastic_std_test))
r2_elastic_std_test = r2_score(y_test_log, y_pred_elastic_std_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_elastic_std_train:<12.4f} {rmse_elastic_std_train:<12.4f} {r2_elastic_std_train:<12.4f}")
print(f"{'test':<10} {mae_elastic_std_test:<12.4f} {rmse_elastic_std_test:<12.4f} {r2_elastic_std_test:<12.4f}")

Epoch 100/200, Loss: 0.1152
Epoch 200/200, Loss: 0.1145
Dataset    MAE          RMSE         R2          
train      0.2463       0.3236       0.4408      
test       0.2479       0.3540       0.3286      


Add all results to our dataframe with metrics on samples.

In [47]:
df_default = pd.DataFrame(metrics_data)
print(df_default.to_string(index=False))

print (" " * 80)

print(f"{'Model':<25} {'Dataset':<8} {'MAE':<12} {'RMSE':<12} {'R2':<12}")

print(f"{'Ridge default':<25} {'train':<8} {mae_ridge_train:<12.4f} {rmse_ridge_train:<12.4f} {r2_ridge_train:<12.4f}")
print(f"{'Ridge default':<25} {'test':<8} {mae_ridge_test:<12.4f} {rmse_ridge_test:<12.4f} {r2_ridge_test:<12.4f}")

print(f"{'Lasso default':<25} {'train':<8} {mae_lasso_train:<12.4f} {rmse_lasso_train:<12.4f} {r2_lasso_train:<12.4f}")
print(f"{'Lasso default':<25} {'test':<8} {mae_lasso_test:<12.4f} {rmse_lasso_test:<12.4f} {r2_lasso_test:<12.4f}")

print(f"{'ElasticNet default':<25} {'train':<8} {mae_elastic_train:<12.4f} {rmse_elastic_train:<12.4f} {r2_elastic_train:<12.4f}")
print(f"{'ElasticNet default':<25} {'test':<8} {mae_elastic_test:<12.4f} {rmse_elastic_test:<12.4f} {r2_elastic_test:<12.4f}")
print(f"{'SGD MinMax':<25} {'train':<8} {mae_sgd_std_train:<12.4f} {rmse_sgd_std_train:<12.4f} {r2_sgd_std_train:<12.4f}")
print(f"{'SGD MinMax':<25} {'test':<8} {mae_sgd_std_test:<12.4f} {rmse_sgd_std_test:<12.4f} {r2_sgd_std_test:<12.4f}")

print(f"{'Ridge MinMax':<25} {'train':<8} {mae_ridge_mm_train:<12.4f} {rmse_ridge_mm_train:<12.4f} {r2_ridge_mm_train:<12.4f}")
print(f"{'Ridge MinMax':<25} {'test':<8} {mae_ridge_mm_test:<12.4f} {rmse_ridge_mm_test:<12.4f} {r2_ridge_mm_test:<12.4f}")

print(f"{'Lasso MinMax':<25} {'train':<8} {mae_lasso_mm_train:<12.4f} {rmse_lasso_mm_train:<12.4f} {r2_lasso_mm_train:<12.4f}")
print(f"{'Lasso MinMax':<25} {'test':<8} {mae_lasso_mm_test:<12.4f} {rmse_lasso_mm_test:<12.4f} {r2_lasso_mm_test:<12.4f}")

print(f"{'ElasticNet MinMax':<25} {'train':<8} {mae_elastic_mm_train:<12.4f} {rmse_elastic_mm_train:<12.4f} {r2_elastic_mm_train:<12.4f}")
print(f"{'ElasticNet MinMax':<25} {'test':<8} {mae_elastic_mm_test:<12.4f} {rmse_elastic_mm_test:<12.4f} {r2_elastic_mm_test:<12.4f}")

print(f"{'SGD Standard':<25} {'train':<8} {mae_sgd_std_train:<12.4f} {rmse_sgd_std_train:<12.4f} {r2_sgd_std_train:<12.4f}")
print(f"{'SGD Standard':<25} {'test':<8} {mae_sgd_std_test:<12.4f} {rmse_sgd_std_test:<12.4f} {r2_sgd_std_test:<12.4f}")

print(f"{'Ridge Standard':<25} {'train':<8} {mae_ridge_std_train:<12.4f} {rmse_ridge_std_train:<12.4f} {r2_ridge_std_train:<12.4f}")
print(f"{'Ridge Standard':<25} {'test':<8} {mae_ridge_std_test:<12.4f} {rmse_ridge_std_test:<12.4f} {r2_ridge_std_test:<12.4f}")

print(f"{'Lasso Standard':<25} {'train':<8} {mae_lasso_std_train:<12.4f} {rmse_lasso_std_train:<12.4f} {r2_lasso_std_train:<12.4f}")
print(f"{'Lasso Standard':<25} {'test':<8} {mae_lasso_std_test:<12.4f} {rmse_lasso_std_test:<12.4f} {r2_lasso_std_test:<12.4f}")

print(f"{'ElasticNet Standard':<25} {'train':<8} {mae_elastic_std_train:<12.4f} {rmse_elastic_std_train:<12.4f} {r2_elastic_std_train:<12.4f}")
print(f"{'ElasticNet Standard':<25} {'test':<8} {mae_elastic_std_test:<12.4f} {rmse_elastic_std_test:<12.4f} {r2_elastic_std_test:<12.4f}")

  model dataset      MAE     RMSE       R2
Our SGD   train 0.234642 0.312862 0.477218
Our SGD    test 0.236528 0.337335 0.390483
Sklearn   train 0.202809 0.277188 0.589640
Sklearn    test 0.204434 0.308484 0.490285
                                                                                
Model                     Dataset  MAE          RMSE         R2          
Ridge default             train    0.2607       0.3445       0.3661      
Ridge default             test     0.2623       0.3521       0.3359      
Lasso default             train    0.2455       0.3241       0.4392      
Lasso default             test     0.2474       0.3494       0.3462      
ElasticNet default        train    0.2426       0.3201       0.4527      
ElasticNet default        test     0.2444       0.3457       0.3599      
SGD MinMax                train    0.2492       0.3269       0.4294      
SGD MinMax                test     0.2507       0.3583       0.3124      
Ridge MinMax              train    0.2

8. Overfitting Models

Let's look at an overfitted model in practice. We know that polynomial regression is easy to overfit. So let's create a toy example and see how regularization works in real life.

Let's create polynomial features of degree 10, remembering that we have only 2 basic features - 'bathrooms' and 'bedrooms'.

In [48]:
x_poly_base = x_train[['bathrooms', 'bedrooms']].values

poly = PolynomialFeatures(degree=10, include_bias=False)
x_poly = poly.fit_transform(x_poly_base)

print(f"Original features: {x_poly_base.shape[1]}")
print(f"Polynomial features: {x_poly.shape[1]}")
print(f"Total objects: {x_poly.shape[0]}")

Original features: 2
Polynomial features: 65
Total objects: 49352


we intentionally overfit the model and see how regularization handles it

In [49]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"x_poly shape: {x_poly.shape}")

x_poly_test = poly.transform(x_test[['bathrooms', 'bedrooms']].values)
print(f"X_poly_test shape: {x_poly_test.shape}")

x_poly shape: (49352, 65)
X_poly_test shape: (74659, 65)


We will also train and fit all our implemented algorithms - Linear Regression,Ridge, Lasso и ElasticNet on a set of polynomial features.

In [50]:
scaler_poly = StandardScaler()
x_poly_scaled = scaler_poly.fit_transform(x_poly)
x_poly_test_scaled = scaler_poly.transform(x_poly_test)

ridge_poly = LinearRegressionRidge(
    learning_rate=0.0001,  # very small steps
    n_iter=200,
    lambda_reg=100.0,
    random_state=21
)

ridge_poly.fit(x_poly_scaled, y_train_log.values)

y_pred_ridge_poly_train = ridge_poly.predict(x_poly_scaled)
y_pred_ridge_poly_test = ridge_poly.predict(x_poly_test_scaled)

mae_ridge_poly_train = mean_absolute_error(y_train_log, y_pred_ridge_poly_train)
rmse_ridge_poly_train = np.sqrt(mean_squared_error(y_train_log, y_pred_ridge_poly_train))
r2_ridge_poly_train = r2_score(y_train_log, y_pred_ridge_poly_train)

mae_ridge_poly_test = mean_absolute_error(y_test_log, y_pred_ridge_poly_test)
rmse_ridge_poly_test = np.sqrt(mean_squared_error(y_test_log, y_pred_ridge_poly_test))
r2_ridge_poly_test = r2_score(y_test_log, y_pred_ridge_poly_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_ridge_poly_train:<12.4f} {rmse_ridge_poly_train:<12.4f} {r2_ridge_poly_train:<12.4f}")
print(f"{'test':<10} {mae_ridge_poly_test:<12.4f} {rmse_ridge_poly_test:<12.4f} {r2_ridge_poly_test:<12.4f}")

Epoch 100/200, Loss: 0.1991
Epoch 200/200, Loss: 1.2762
Dataset    MAE          RMSE         R2          
train      0.3137       0.4161       0.0754      
test       51.2731      13923.8422   -1038440199.7415


In [51]:
lasso_poly = LinearRegressionLasso(
    learning_rate=0.0001,
    n_iter=200,
    lambda_reg=10.0,
    random_state=21
)

lasso_poly.fit(x_poly_scaled, y_train_log.values)

y_pred_lasso_poly_train = lasso_poly.predict(x_poly_scaled)
y_pred_lasso_poly_test = lasso_poly.predict(x_poly_test_scaled)

mae_lasso_poly_train = mean_absolute_error(y_train_log, y_pred_lasso_poly_train)
rmse_lasso_poly_train = np.sqrt(mean_squared_error(y_train_log, y_pred_lasso_poly_train))
r2_lasso_poly_train = r2_score(y_train_log, y_pred_lasso_poly_train)

mae_lasso_poly_test = mean_absolute_error(y_test_log, y_pred_lasso_poly_test)
rmse_lasso_poly_test = np.sqrt(mean_squared_error(y_test_log, y_pred_lasso_poly_test))
r2_lasso_poly_test = r2_score(y_test_log, y_pred_lasso_poly_test)

print(f"{'Dataset':<10} {'MAE':<12} {'RMSE':<12} {'R2':<12}")
print(f"{'train':<10} {mae_lasso_poly_train:<12.4f} {rmse_lasso_poly_train:<12.4f} {r2_lasso_poly_train:<12.4f}")
print(f"{'test':<10} {mae_lasso_poly_test:<12.4f} {rmse_lasso_poly_test:<12.4f} {r2_lasso_poly_test:<12.4f}")

Epoch 100/200, Loss: 0.5323
Epoch 200/200, Loss: 0.5355
Dataset    MAE          RMSE         R2          
train      0.3243       0.4322       0.0023      
test       21032.4593   5746777.6141 -176893586208675.1250


In [52]:
elastic_poly = LinearRegressionElasticNet(
    learning_rate=0.0001,
    n_iter=200,
    lambda_1=10.0,
    lambda_2=10.0,
    random_state=21
)

elastic_poly.fit(x_poly_scaled, y_train_log.values)

y_pred_elastic_poly_train = elastic_poly.predict(x_poly_scaled)
y_pred_elastic_poly_test = elastic_poly.predict(x_poly_test_scaled)

mae_elastic_poly_train = mean_absolute_error(y_train_log, y_pred_elastic_poly_train)
rmse_elastic_poly_train = np.sqrt(mean_squared_error(y_train_log, y_pred_elastic_poly_train))
r2_elastic_poly_train = r2_score(y_train_log, y_pred_elastic_poly_train)

mae_elastic_poly_test = mean_absolute_error(y_test_log, y_pred_elastic_poly_test)
rmse_elastic_poly_test = np.sqrt(mean_squared_error(y_test_log, y_pred_elastic_poly_test))
r2_elastic_poly_test = r2_score(y_test_log, y_pred_elastic_poly_test)

print(f"{'Dataset':<10} {'MAE':<18} {'RMSE':<18} {'R2':<18}")
print(f"{'train':<10} {mae_elastic_poly_train:<18.4f} {rmse_elastic_poly_train:<18.4f} {r2_elastic_poly_train:<18.4f}")
print(f"{'test':<10} {mae_elastic_poly_test:<18.4f} {rmse_elastic_poly_test:<18.4f} {r2_elastic_poly_test:<18.4f}")

Epoch 100/200, Loss: 0.5367
Epoch 200/200, Loss: 0.5320
Dataset    MAE                RMSE               R2                
train      0.3235             0.4309             0.0082            
test       38116.7649         10414858.6185      -580991878347060.2500


Save the quality metrics results in the result dataframe.

In [53]:
print(f"{'Model':<30} {'Dataset':<8} {'MAE':<20} {'RMSE':<20} {'R2':<20}")

print(f"{'Ridge default':<30} {'train':<8} {mae_ridge_train:<20.4f} {rmse_ridge_train:<20.4f} {r2_ridge_train:<20.4f}")
print(f"{'Ridge default':<30} {'test':<8} {mae_ridge_test:<20.4f} {rmse_ridge_test:<20.4f} {r2_ridge_test:<20.4f}")

print(f"{'Lasso default':<30} {'train':<8} {mae_lasso_train:<20.4f} {rmse_lasso_train:<20.4f} {r2_lasso_train:<20.4f}")
print(f"{'Lasso default':<30} {'test':<8} {mae_lasso_test:<20.4f} {rmse_lasso_test:<20.4f} {r2_lasso_test:<20.4f}")

print(f"{'ElasticNet default':<30} {'train':<8} {mae_elastic_train:<20.4f} {rmse_elastic_train:<20.4f} {r2_elastic_train:<20.4f}")
print(f"{'ElasticNet default':<30} {'test':<8} {mae_elastic_test:<20.4f} {rmse_elastic_test:<20.4f} {r2_elastic_test:<20.4f}")

print(f"{'SGD MinMax':<30} {'train':<8} {mae_sgd_std_train:<20.4f} {rmse_sgd_std_train:<20.4f} {r2_sgd_std_train:<20.4f}")
print(f"{'SGD MinMax':<30} {'test':<8} {mae_sgd_std_test:<20.4f} {rmse_sgd_std_test:<20.4f} {r2_sgd_std_test:<20.4f}")

print(f"{'Ridge MinMax':<30} {'train':<8} {mae_ridge_mm_train:<20.4f} {rmse_ridge_mm_train:<20.4f} {r2_ridge_mm_train:<20.4f}")
print(f"{'Ridge MinMax':<30} {'test':<8} {mae_ridge_mm_test:<20.4f} {rmse_ridge_mm_test:<20.4f} {r2_ridge_mm_test:<20.4f}")

print(f"{'Lasso MinMax':<30} {'train':<8} {mae_lasso_mm_train:<20.4f} {rmse_lasso_mm_train:<20.4f} {r2_lasso_mm_train:<20.4f}")
print(f"{'Lasso MinMax':<30} {'test':<8} {mae_lasso_mm_test:<20.4f} {rmse_lasso_mm_test:<20.4f} {r2_lasso_mm_test:<20.4f}")

print(f"{'ElasticNet MinMax':<30} {'train':<8} {mae_elastic_mm_train:<20.4f} {rmse_elastic_mm_train:<20.4f} {r2_elastic_mm_train:<20.4f}")
print(f"{'ElasticNet MinMax':<30} {'test':<8} {mae_elastic_mm_test:<20.4f} {rmse_elastic_mm_test:<20.4f} {r2_elastic_mm_test:<20.4f}")

print(f"{'SGD Standard':<30} {'train':<8} {mae_sgd_std_train:<20.4f} {rmse_sgd_std_train:<20.4f} {r2_sgd_std_train:<20.4f}")
print(f"{'SGD Standard':<30} {'test':<8} {mae_sgd_std_test:<20.4f} {rmse_sgd_std_test:<20.4f} {r2_sgd_std_test:<20.4f}")

print(f"{'Ridge Standard':<30} {'train':<8} {mae_ridge_std_train:<20.4f} {rmse_ridge_std_train:<20.4f} {r2_ridge_std_train:<20.4f}")
print(f"{'Ridge Standard':<30} {'test':<8} {mae_ridge_std_test:<20.4f} {rmse_ridge_std_test:<20.4f} {r2_ridge_std_test:<20.4f}")

print(f"{'Lasso Standard':<30} {'train':<8} {mae_lasso_std_train:<20.4f} {rmse_lasso_std_train:<20.4f} {r2_lasso_std_train:<20.4f}")
print(f"{'Lasso Standard':<30} {'test':<8} {mae_lasso_std_test:<20.4f} {rmse_lasso_std_test:<20.4f} {r2_lasso_std_test:<20.4f}")

print(f"{'ElasticNet Standard':<30} {'train':<8} {mae_elastic_std_train:<20.4f} {rmse_elastic_std_train:<20.4f} {r2_elastic_std_train:<20.4f}")
print(f"{'ElasticNet Standard':<30} {'test':<8} {mae_elastic_std_test:<20.4f} {rmse_elastic_std_test:<20.4f} {r2_elastic_std_test:<20.4f}")

print(f"{'Ridge Polynomial':<30} {'train':<8} {mae_ridge_poly_train:<20.4f} {rmse_ridge_poly_train:<20.4f} {r2_ridge_poly_train:<20.4f}")
print(f"{'Ridge Polynomial':<30} {'test':<8} {mae_ridge_poly_test:<20.4f} {rmse_ridge_poly_test:<20.4f} {r2_ridge_poly_test:<20.4f}")

print(f"{'Lasso Polynomial':<30} {'train':<8} {mae_lasso_poly_train:<20.4f} {rmse_lasso_poly_train:<20.4f} {r2_lasso_poly_train:<20.4f}")
print(f"{'Lasso Polynomial':<30} {'test':<8} {mae_lasso_poly_test:<20.4f} {rmse_lasso_poly_test:<20.4f} {r2_lasso_poly_test:<20.4f}")

print(f"{'ElasticNet Polynomial':<30} {'train':<8} {mae_elastic_poly_train:<20.4f} {rmse_elastic_poly_train:<20.4f} {r2_elastic_poly_train:<20.4f}")
print(f"{'ElasticNet Polynomial':<30} {'test':<8} {mae_elastic_poly_test:<20.4f} {rmse_elastic_poly_test:<20.4f} {r2_elastic_poly_test:<20.4f}")


Model                          Dataset  MAE                  RMSE                 R2                  
Ridge default                  train    0.2607               0.3445               0.3661              
Ridge default                  test     0.2623               0.3521               0.3359              
Lasso default                  train    0.2455               0.3241               0.4392              
Lasso default                  test     0.2474               0.3494               0.3462              
ElasticNet default             train    0.2426               0.3201               0.4527              
ElasticNet default             test     0.2444               0.3457               0.3599              
SGD MinMax                     train    0.2492               0.3269               0.4294              
SGD MinMax                     test     0.2507               0.3583               0.3124              
Ridge MinMax                   train    0.2734               0.3742      

The best is Lasso.

9. Native models

Calculate the mean and median and add the results to the final dataframe.

In [54]:
y_mean = np.full_like(y_test_log, y_train_log.mean())
y_median = np.full_like(y_test_log, y_train_log.median())

mae_mean = mean_absolute_error(y_test_log, y_mean)
rmse_mean = np.sqrt(mean_squared_error(y_test_log, y_mean))
r2_mean = r2_score(y_test_log, y_mean)

mae_median = mean_absolute_error(y_test_log, y_median)
rmse_median = np.sqrt(mean_squared_error(y_test_log, y_median))
r2_median = r2_score(y_test_log, y_median)

print(f"{'Naive mean':<30} {'test':<8} {mae_mean:<20.4f} {rmse_mean:<20.4f} {r2_mean:<20.4f}")
print(f"{'Naive median':<30} {'test':<8} {mae_median:<20.4f} {rmse_median:<20.4f} {r2_median:<20.4f}")

Naive mean                     test     0.3245               0.4321               -0.0000             
Naive median                   test     0.3223               0.4342               -0.0097             


Compare the results

Print your final tables.
Which model is better?
Which model is the most stable?

In [55]:
print(f"{'Model':<30} {'Dataset':<8} {'MAE':<20} {'RMSE':<20} {'R2':<20}")

print(f"{'Ridge default':<30} {'train':<8} {mae_ridge_train:<20.4f} {rmse_ridge_train:<20.4f} {r2_ridge_train:<20.4f}")
print(f"{'Ridge default':<30} {'test':<8} {mae_ridge_test:<20.4f} {rmse_ridge_test:<20.4f} {r2_ridge_test:<20.4f}")

print(f"{'Lasso default':<30} {'train':<8} {mae_lasso_train:<20.4f} {rmse_lasso_train:<20.4f} {r2_lasso_train:<20.4f}")
print(f"{'Lasso default':<30} {'test':<8} {mae_lasso_test:<20.4f} {rmse_lasso_test:<20.4f} {r2_lasso_test:<20.4f}")

print(f"{'ElasticNet default':<30} {'train':<8} {mae_elastic_train:<20.4f} {rmse_elastic_train:<20.4f} {r2_elastic_train:<20.4f}")
print(f"{'ElasticNet default':<30} {'test':<8} {mae_elastic_test:<20.4f} {rmse_elastic_test:<20.4f} {r2_elastic_test:<20.4f}")

print(f"{'SGD MinMax':<30} {'train':<8} {mae_sgd_std_train:<20.4f} {rmse_sgd_std_train:<20.4f} {r2_sgd_std_train:<20.4f}")
print(f"{'SGD MinMax':<30} {'test':<8} {mae_sgd_std_test:<20.4f} {rmse_sgd_std_test:<20.4f} {r2_sgd_std_test:<20.4f}")

print(f"{'Ridge MinMax':<30} {'train':<8} {mae_ridge_mm_train:<20.4f} {rmse_ridge_mm_train:<20.4f} {r2_ridge_mm_train:<20.4f}")
print(f"{'Ridge MinMax':<30} {'test':<8} {mae_ridge_mm_test:<20.4f} {rmse_ridge_mm_test:<20.4f} {r2_ridge_mm_test:<20.4f}")

print(f"{'Lasso MinMax':<30} {'train':<8} {mae_lasso_mm_train:<20.4f} {rmse_lasso_mm_train:<20.4f} {r2_lasso_mm_train:<20.4f}")
print(f"{'Lasso MinMax':<30} {'test':<8} {mae_lasso_mm_test:<20.4f} {rmse_lasso_mm_test:<20.4f} {r2_lasso_mm_test:<20.4f}")

print(f"{'ElasticNet MinMax':<30} {'train':<8} {mae_elastic_mm_train:<20.4f} {rmse_elastic_mm_train:<20.4f} {r2_elastic_mm_train:<20.4f}")
print(f"{'ElasticNet MinMax':<30} {'test':<8} {mae_elastic_mm_test:<20.4f} {rmse_elastic_mm_test:<20.4f} {r2_elastic_mm_test:<20.4f}")

print(f"{'SGD Standard':<30} {'train':<8} {mae_sgd_std_train:<20.4f} {rmse_sgd_std_train:<20.4f} {r2_sgd_std_train:<20.4f}")
print(f"{'SGD Standard':<30} {'test':<8} {mae_sgd_std_test:<20.4f} {rmse_sgd_std_test:<20.4f} {r2_sgd_std_test:<20.4f}")

print(f"{'Ridge Standard':<30} {'train':<8} {mae_ridge_std_train:<20.4f} {rmse_ridge_std_train:<20.4f} {r2_ridge_std_train:<20.4f}")
print(f"{'Ridge Standard':<30} {'test':<8} {mae_ridge_std_test:<20.4f} {rmse_ridge_std_test:<20.4f} {r2_ridge_std_test:<20.4f}")

print(f"{'Lasso Standard':<30} {'train':<8} {mae_lasso_std_train:<20.4f} {rmse_lasso_std_train:<20.4f} {r2_lasso_std_train:<20.4f}")
print(f"{'Lasso Standard':<30} {'test':<8} {mae_lasso_std_test:<20.4f} {rmse_lasso_std_test:<20.4f} {r2_lasso_std_test:<20.4f}")

print(f"{'ElasticNet Standard':<30} {'train':<8} {mae_elastic_std_train:<20.4f} {rmse_elastic_std_train:<20.4f} {r2_elastic_std_train:<20.4f}")
print(f"{'ElasticNet Standard':<30} {'test':<8} {mae_elastic_std_test:<20.4f} {rmse_elastic_std_test:<20.4f} {r2_elastic_std_test:<20.4f}")

print(f"{'Ridge Polynomial':<30} {'train':<8} {mae_ridge_poly_train:<20.4f} {rmse_ridge_poly_train:<20.4f} {r2_ridge_poly_train:<20.4f}")
print(f"{'Ridge Polynomial':<30} {'test':<8} {mae_ridge_poly_test:<20.4f} {rmse_ridge_poly_test:<20.4f} {r2_ridge_poly_test:<20.4f}")

print(f"{'Lasso Polynomial':<30} {'train':<8} {mae_lasso_poly_train:<20.4f} {rmse_lasso_poly_train:<20.4f} {r2_lasso_poly_train:<20.4f}")
print(f"{'Lasso Polynomial':<30} {'test':<8} {mae_lasso_poly_test:<20.4f} {rmse_lasso_poly_test:<20.4f} {r2_lasso_poly_test:<20.4f}")

print(f"{'ElasticNet Polynomial':<30} {'train':<8} {mae_elastic_poly_train:<20.4f} {rmse_elastic_poly_train:<20.4f} {r2_elastic_poly_train:<20.4f}")
print(f"{'ElasticNet Polynomial':<30} {'test':<8} {mae_elastic_poly_test:<20.4f} {rmse_elastic_poly_test:<20.4f} {r2_elastic_poly_test:<20.4f}")

print(f"{'Naive mean':<30} {'test':<8} {mae_mean:<20.4f} {rmse_mean:<20.4f} {r2_mean:<20.4f}")
print(f"{'Naive median':<30} {'test':<8} {mae_median:<20.4f} {rmse_median:<20.4f} {r2_median:<20.4f}")


Model                          Dataset  MAE                  RMSE                 R2                  
Ridge default                  train    0.2607               0.3445               0.3661              
Ridge default                  test     0.2623               0.3521               0.3359              
Lasso default                  train    0.2455               0.3241               0.4392              
Lasso default                  test     0.2474               0.3494               0.3462              
ElasticNet default             train    0.2426               0.3201               0.4527              
ElasticNet default             test     0.2444               0.3457               0.3599              
SGD MinMax                     train    0.2492               0.3269               0.4294              
SGD MinMax                     test     0.2507               0.3583               0.3124              
Ridge MinMax                   train    0.2734               0.3742      

The best and most stable model is Lasso with MinMaxScaler